# Excluding Time Segments from the GLM Fit with Masks

Recordings often contain segments that should not influence the model fit: motion
artifacts, periods in which an optode lost contact, or samples that are missing
altogether. Removing these samples from the time series is not an option, because the
GLM relies on the temporal structure of the data: the design matrix is defined on the
full time axis and the noise models account for temporal correlations.

Instead, `glm.fit` accepts a boolean `mask` that marks which samples enter the fit.
It follows the convention of the masks in `cedalion.sigproc.quality`:

- `True` (`quality.CLEAN`): the sample is used,
- `False` (`quality.TAINTED`): the sample is ignored.

The fit is determined by the unmasked samples only. The design matrix is not masked,
so `glm.predict` still provides a model prediction in the masked segments.

This notebook demonstrates masking on simulated data with a known ground truth.

In [ ]:
# This cells setups the environment when executed in Google Colab.
try:
    import google.colab
    !curl -s https://raw.githubusercontent.com/ibs-lab/cedalion/dev/scripts/colab_setup.py -o colab_setup.py
    # Select branch with --branch "branch name" (default is "dev")
    %run colab_setup.py
except ImportError:
    pass

In [ ]:
import matplotlib.pyplot as p
import numpy as np
import pandas as pd
import xarray as xr

import cedalion
import cedalion.dataclasses as cdc
import cedalion.models.glm as glm
import cedalion.sigproc.quality as quality
import cedalion.vis.blocks as vbx
import cedalion.vis.colors
import cedalion.xrutils as xrutils
from cedalion import units

xr.set_options(display_expand_data=False);

## Simulated data

### Time series with a known ground truth

We simulate two channels with the same hemodynamic response to five trials of
'StimA' and a constant offset. The HRF regressor is normalized to a peak value of 1,
so that the betas are the response amplitudes in µM.

In [ ]:
rng = np.random.default_rng(42)

fs = 10.0 * units.Hz  # sampling rate
T = 240 * units.s  # time series length
nsample = int(T * fs)

ts = cdc.build_timeseries(
    rng.normal(0, 0.1, (nsample, 2, 2)),
    dims=["time", "channel", "chromo"],
    time=np.arange(nsample) / fs.magnitude,
    channel=["S1D1", "S1D2"],
    value_units=units.uM,
    time_units=units.s,
    other_coords={"chromo": ["HbO", "HbR"]},
)

stim = pd.DataFrame(
    {
        "onset": [20.0, 60.0, 100.0, 140.0, 180.0],
        "trial_type": "StimA",
        "value": 1.0,
        "duration": 10.0,
    }
)

dms = glm.design_matrix.hrf_regressors(
    ts, stim, glm.Gamma(tau=0 * units.s, sigma=5 * units.s)
) & glm.design_matrix.drift_regressors(ts, drift_order=0)

# normalize the HRF regressor to a peak value of 1
hrf = dms.common.sel(regressor="HRF StimA")
dms.common.loc[{"regressor": "HRF StimA"}] = hrf / hrf.max("time")

# ground truth: response amplitude and offset in µM
true_betas = xr.DataArray(
    [[1.0, 0.5], [-0.3, 0.2]],
    dims=["chromo", "regressor"],
    coords={"chromo": ["HbO", "HbR"], "regressor": ["HRF StimA", "Drift 0"]},
).expand_dims(channel=ts.channel.values)

ts += xr.dot(dms.common, true_betas, dim="regressor").pint.quantify("uM")

### Corrupting the time series

We add two problems that frequently occur in real recordings:

- Between 100 s and 115 s, channel S1D1 contains a large artifact in both
  chromophores, e.g. from a movement of the probe. This segment overlaps with the
  third trial.
- Between 170 s and 175 s, the HbR signal of channel S1D2 is missing (NaN).

The mask marks both segments as `TAINTED`. `xrutils.mask` creates a mask with the
shape of the time series, in which all samples are `CLEAN`. Note that the mask of
S1D2 only excludes HbR, while the HbO samples of that segment remain in the fit.

In [ ]:
artifact = ((ts.time >= 100) & (ts.time < 115)).values
dropout = ((ts.time >= 170) & (ts.time < 175)).values

bump = np.sin(np.pi * (ts.time.values[artifact] - 100) / 15)
ts.loc[{"channel": "S1D1", "chromo": "HbO", "time": artifact}] += 3 * bump * units.uM
ts.loc[{"channel": "S1D1", "chromo": "HbR", "time": artifact}] += 1.5 * bump * units.uM
ts.loc[{"channel": "S1D2", "chromo": "HbR", "time": dropout}] = np.nan * units.uM

mask = xrutils.mask(ts, quality.CLEAN)
mask.loc[{"channel": "S1D1", "time": artifact}] = quality.TAINTED
mask.loc[{"channel": "S1D2", "chromo": "HbR", "time": dropout}] = quality.TAINTED

display(mask)

In [ ]:
# masked segments are shaded in gray
SEGMENT_FMT = {"ec": "none", "fc": "0.5", "alpha": 0.3}

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 5))
for i, ch in enumerate(ts.channel.values):
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbO"), "r-", lw=1, label="HbO")
    ax[i].plot(ts.time, ts.sel(channel=ch, chromo="HbR"), "b-", lw=1, label="HbR")
    vbx.plot_stim_markers(ax[i], stim)
    segments = quality.mask_to_segments(mask.sel(channel=ch).all("chromo"))
    vbx.plot_segments(ax[i], segments, fmt=SEGMENT_FMT, label="masked")
    ax[i].set_title(f"Ch. {ch}")
    ax[i].set_ylabel(r"$\Delta$ c / µM")
    ax[i].grid(True, alpha=0.3)
ax[0].legend(loc="upper left", ncol=3)
ax[1].set_xlabel("time / s")
p.tight_layout()

### Unmasked NaNs are an error

NaNs in the time series would invalidate the fit of the affected channel. Instead of
silently returning NaN results, `glm.fit` raises an exception if the time series
contains NaNs (or infs) in samples that are not masked. This catches errors in the
preprocessing early.

In [ ]:
try:
    glm.fit(ts, dms, noise_model="ols")
except ValueError as e:
    print(e)

### Fitting with a mask

The error message suggests `mask=np.isfinite(ts)`, which excludes exactly the
non-finite samples. This removes the NaNs, but the artifact in S1D1 remains in the
fit. We compare this fit with a fit that uses our mask, which excludes both segments.

The GLM models each sample of the time series as a weighted sum of the regressors at
the same time point, i.e. each sample corresponds to one row of the design matrix.
A mask removes samples together with their rows of the design matrix. The plot shows
what enters the fit of S1D1 (HbO) with our mask.

In [ ]:
result_finite = glm.fit(ts, dms, noise_model="ols", mask=np.isfinite(ts))
result_masked = glm.fit(ts, dms, noise_model="ols", mask=mask)

ch, chromo = "S1D1", "HbO"
used = mask.sel(channel=ch, chromo=chromo)
y = ts.sel(channel=ch, chromo=chromo)
segments = quality.mask_to_segments(used)

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 5), height_ratios=[3, 2])
ax[0].plot(ts.time, y.where(used), "r-", lw=1, label="used in the fit")
ax[0].plot(ts.time, y.where(~used), "-", c="0.6", lw=1, label="masked")
ax[0].set_ylabel(r"$\Delta$ HbO / µM")
ax[0].set_title(f"Ch. {ch}, HbO: data")

for regressor, c in zip(dms.common.regressor.values, ["#2a78d6", "#1baf7a"]):
    x = dms.common.sel(regressor=regressor, chromo=chromo)
    ax[1].plot(ts.time, x.where(used), "-", c=c, lw=2, label=regressor)
    ax[1].plot(ts.time, x.where(~used), "-", c=c, lw=2, alpha=0.3)
ax[1].set_title("design matrix")
ax[1].set_xlabel("time / s")

for a in ax:
    vbx.plot_segments(a, segments, fmt=SEGMENT_FMT)
    a.grid(True, alpha=0.3)
    a.legend(loc="upper left", ncol=2)
p.tight_layout()

### Model predictions and residuals in masked segments

The prediction is the weighted sum of the regressors, with the estimated betas as
weights. Because the design matrix is not masked, the prediction is defined at all
time points. The plot compares the predictions of both fits for S1D1 (HbO), with the
estimated HRF amplitudes in the legend (ground truth: 1 µM).

Without the mask, the artifact pulls the fit upward, and the response is
overestimated. With the mask, the model predicts the response to the third trial,
although the data in this segment were ignored. The residuals (`.sm.resid`) are NaN
in the masked segment, since there is no data point that the model is compared to.
They are labeled with the sample index of the time series.

In [ ]:
def hrf_beta(result):
    return result.sm.params.sel(channel=ch, chromo="HbO", regressor="HRF StimA").item()


pred_finite = glm.predict(ts, result_finite.sm.params, dms)
pred_masked = glm.predict(ts, result_masked.sm.params, dms)
resid = result_masked.sm.resid

ch = "S1D1"
segments = quality.mask_to_segments(mask.sel(channel=ch, chromo="HbO"))

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 6), height_ratios=[2, 1])
ax[0].plot(ts.time, ts.sel(channel=ch, chromo="HbO"), "-", c="0.6", lw=1, label="data")
ax[0].plot(
    pred_finite.time,
    pred_finite.sel(channel=ch, chromo="HbO"),
    "--",
    c="#eb6834",
    lw=2,
    label=f"model, mask=np.isfinite(ts) (HRF beta {hrf_beta(result_finite):.2f} µM)",
)
ax[0].plot(
    pred_masked.time,
    pred_masked.sel(channel=ch, chromo="HbO"),
    "-",
    c="#2a78d6",
    lw=2,
    label=f"model, mask=mask (HRF beta {hrf_beta(result_masked):.2f} µM)",
)
ax[0].set_ylabel(r"$\Delta$ HbO / µM")
ax[0].set_title(f"Ch. {ch}, HbO")

ax[1].plot(
    ts.time.values[resid.resid.values],
    resid.sel(channel=ch, chromo="HbO"),
    "-",
    c="0.3",
    lw=1,
)
ax[1].axhline(0, c="0.5", lw=1)
ax[1].set_ylabel("residual / µM")
ax[1].set_xlabel("time / s")
ax[1].set_title("residuals of the fit with mask=mask")

for a in ax:
    vbx.plot_segments(a, segments, fmt=SEGMENT_FMT, label="masked")
    a.grid(True, alpha=0.3)
ax[0].legend(loc="upper left", ncol=2)
ax[1].set_xlim(50, 170)
p.tight_layout()

### Mask shapes

The mask must have a time dimension. All other dimensions are optional and must be
dimensions of the time series. Missing dimensions are broadcast:

- a mask with dimension *time* masks a segment in all channels and chromophores,
- a mask with dimensions *time* and *channel* masks segments per channel, in all
  chromophores,
- a mask with dimensions *time*, *channel* and *chromo* masks segments per channel
  and chromophore.

For image reconstructions, *channel* is replaced by *parcel* or *vertex*. The plot
shows how each mask applies to the time series (red: `TAINTED`, blue: `CLEAN`).

In [ ]:
artifact_da = xr.DataArray(artifact, dims="time", coords={"time": ts.time})

masks = {
    "time": ~artifact_da,
    "time x channel": ~(artifact_da & (ts.channel == "S1D1")),
    "time x channel x chromo": ~(
        artifact_da & (ts.channel == "S1D1") & (ts.chromo == "HbO")
    ),
}

norm, cmap = cedalion.vis.colors.mask_cmap()
f, ax = p.subplots(1, 3, sharey=True, figsize=(12, 3.2))
for a, (name, m) in zip(ax, masks.items()):
    applied = (
        m.broadcast_like(mask)
        .stack(row=["channel", "chromo"])
        .transpose("row", "time")
    )
    im = a.pcolormesh(
        applied.time,
        np.arange(applied.sizes["row"]),
        applied,
        norm=norm,
        cmap=cmap,
        shading="nearest",
    )
    a.set_yticks(np.arange(applied.sizes["row"]))
    a.set_yticklabels([f"{c} {h}" for c, h in applied.row.values])
    a.set_title(f"mask dims:\n{name}")
    a.set_xlabel("time / s")
ax[0].invert_yaxis()
cb = f.colorbar(im, ax=ax, ticks=[0.25, 0.75])
cb.set_ticklabels(["TAINTED", "CLEAN"])

Masks can be combined with the usual boolean operators. Here, each mask is combined
with `np.isfinite(ts)` to also exclude the missing HbR samples of S1D2. The plot shows
the predictions for S1D1. The artifact affects both chromophores of S1D1, so the
mask with the chromo dimension, which excludes only HbO, leaves the artifact in the
HbR fit. The other two masks give the same predictions for S1D1, so their lines
overlap.

In [ ]:
f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 6))
for a, chromo in zip(ax, ["HbO", "HbR"]):
    a.plot(ts.time, ts.sel(channel="S1D1", chromo=chromo), "-", c="0.6", lw=1)
    a.set_ylabel(rf"$\Delta$ {chromo} / µM")
    a.set_title(f"Ch. S1D1, {chromo}")

styles = [("#2a78d6", "-"), ("#eb6834", "--"), ("#1baf7a", "-")]
for (name, m), (c, ls) in zip(masks.items(), styles):
    result = glm.fit(ts, dms, noise_model="ols", mask=m & np.isfinite(ts))
    pred = glm.predict(ts, result.sm.params, dms)
    for a, chromo in zip(ax, ["HbO", "HbR"]):
        a.plot(
            pred.time,
            pred.sel(channel="S1D1", chromo=chromo),
            ls,
            c=c,
            lw=2,
            label=f"mask dims: {name}",
        )

artifact_segment = quality.mask_to_segments(~artifact_da)
for a in ax:
    vbx.plot_segments(a, artifact_segment, fmt=SEGMENT_FMT, label="artifact")
    a.grid(True, alpha=0.3)
ax[0].legend(loc="upper left", ncol=2)
ax[1].set_xlabel("time / s")
ax[1].set_xlim(50, 170)
p.tight_layout()

### NaNs in the design matrix

Regressors can contain NaNs, too. For example, a short-channel regressor contains the
NaNs of the short channel it was built from. We simulate two short channels, S1D9 and
S2D9, which measure a physiological signal with a different rhythm each. The signal
of S1D9 is added to the long channel S1D1 and that of S2D9 to S1D2. Then we remove
the samples of S1D9 between 200 s and 210 s.

In [ ]:
true_short = 1.0  # ground truth beta of the short-channel regressor

phase = rng.uniform(0, 2 * np.pi, (1, 2, 2))
freq = np.array([0.1, 0.27])[None, :, None]  # one rhythm per short channel / Hz
ts_short = cdc.build_timeseries(
    0.3 * np.sin(2 * np.pi * freq * ts.time.values[:, None, None] + phase)
    + rng.normal(0, 0.05, (nsample, 2, 2)),
    dims=["time", "channel", "chromo"],
    time=ts.time.values,
    channel=["S1D9", "S2D9"],
    value_units=units.uM,
    time_units=units.s,
    other_coords={"chromo": ["HbO", "HbR"]},
)

# S1D9 -> S1D1, S2D9 -> S1D2
ts_ss = ts + true_short * ts_short.assign_coords(channel=ts.channel.values)

gap = ((ts.time >= 200) & (ts.time < 210)).values
ts_short.loc[{"channel": "S1D9", "time": gap}] = np.nan * units.uM

`max_corr_short_channel_regressor` assigns to each long channel the short channel
with the highest correlation. Each pair of channels is correlated over the samples
at which both are finite, so S1D9 is still assigned to S1D1 (see the titles of the
next plot). Its NaNs end up in the channel-wise regressor of S1D1, and `glm.fit`
raises an exception.

In [ ]:
dms_ss = dms & glm.design_matrix.max_corr_short_channel_regressor(ts_ss, ts_short)
try:
    glm.fit(ts_ss, dms_ss, noise_model="ols", mask=mask & np.isfinite(ts_ss))
except ValueError as e:
    print(e)

`DesignMatrix.fill_nonfinite` replaces the non-finite values of the regressors (with
0 by default) and returns a mask with the dimensions of the time series, in which the
affected samples are `TAINTED`. A NaN in a channel-wise regressor masks only the
channel that uses this regressor, here S1D1. A NaN in a regressor of the common part
of the design matrix, e.g. from `average_short_channel_regressor`, masks all channels.

In [ ]:
dms_filled, dm_mask = dms_ss.fill_nonfinite(ts_ss)

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 4))
for i, ch in enumerate(ts.channel.values):
    regressor = dms_filled.channel_wise[0].sel(channel=ch, chromo="HbO")
    ax[i].plot(ts.time, regressor.squeeze("regressor"), "r-", lw=1)
    segments = quality.mask_to_segments(dm_mask.sel(channel=ch).all("chromo"))
    vbx.plot_segments(ax[i], segments, fmt=SEGMENT_FMT, label="masked")
    short_ch = regressor.short_channel.item()
    ax[i].set_title(f"Ch. {ch}, HbO: short-channel regressor ({short_ch}), filled")
    ax[i].set_ylabel(r"$\Delta$ c / µM")
    ax[i].grid(True, alpha=0.3)
ax[1].set_xlabel("time / s")
ax[1].set_xlim(150, 240)
p.tight_layout()

The returned mask is combined with the other masks. The plot shows the fit of S1D1
(HbO) around the gap, split into the contributions of the regressors (beta times
regressor). In the gap, the short-channel contribution is the fill value 0, so the
prediction there lacks the physiological signal. These samples don't enter the fit,
so the betas are not affected. The prediction in the gap is drawn dashed, because it
depends on the fill value. `pred_ss.where(dm_mask)` removes these samples from the
prediction.

In [ ]:
mask_ss = mask & np.isfinite(ts_ss) & dm_mask
result_ss = glm.fit(ts_ss, dms_filled, noise_model="ols", mask=mask_ss)
pred_ss = glm.predict(ts_ss, result_ss.sm.params, dms_filled)

ch, chromo = "S1D1", "HbO"
window = {"channel": ch, "chromo": chromo, "time": slice(150, 240)}
betas = result_ss.sm.params.sel(channel=ch, chromo=chromo)
regressors = {
    "HRF StimA": dms_filled.common.sel(
        time=window["time"], chromo=chromo, regressor="HRF StimA"
    ),
    "short": dms_filled.channel_wise[0]
    .sel({**window, "regressor": "short"})
    .pint.dequantify(),
}
used = mask_ss.sel(window)
known = dm_mask.sel(window)
segments = quality.mask_to_segments(used)

f, ax = p.subplots(2, 1, sharex=True, figsize=(12, 6), height_ratios=[3, 2])
y = ts_ss.sel(window)
ax[0].plot(y.time, y.where(used), "r-", lw=1, label="used in the fit")
ax[0].plot(y.time, y.where(~used), "-", c="0.6", lw=1, label="masked")
pred = pred_ss.sel(window)
ax[0].plot(y.time, pred.where(known), "k-", lw=2, label="model")
ax[0].plot(y.time, pred.where(~known), "--", c="0.3", lw=1.5, label="model with fill value")
ax[0].set_ylabel(r"$\Delta$ HbO / µM")
ax[0].set_title(f"Ch. {ch}, HbO: data and model")

for (name, regressor), c in zip(regressors.items(), ["#2a78d6", "#eb6834"]):
    beta = betas.sel(regressor=name).item()
    ax[1].plot(
        regressor.time, beta * regressor, "-", c=c, lw=2, label=f"{beta:.2f} × {name}"
    )
ax[1].set_ylabel(r"$\Delta$ HbO / µM")
ax[1].set_title("contributions of the regressors (without drift)")
ax[1].set_xlabel("time / s")

for a in ax:
    vbx.plot_segments(a, segments, fmt=SEGMENT_FMT)
    a.grid(True, alpha=0.3)
    a.legend(loc="upper left", ncol=4)
p.tight_layout()

### Subtracting nuisance signals

To remove the nuisance signals from the data, we predict the time series with the
betas of the nuisance regressors only and subtract this prediction. In the gap, the
short-channel regressor is unknown, so the physiological signal can't be removed
there. `where(dm_mask)` sets these samples to NaN. Without it, the cleaned signal
would still contain the physiological signal in the gap, and nothing would indicate
this. The cleaned signal of S1D1 matches the true HRF outside of the gap.

In [ ]:
betas = result_ss.sm.params
is_hrf = betas.regressor.str.startswith("HRF")
nuisance = glm.predict(ts_ss, betas.where(~is_hrf, 0), dms_filled)
cleaned = (ts_ss - nuisance.pint.quantify(ts_ss.pint.units)).where(dm_mask)

true_hrf = true_betas.sel(regressor="HRF StimA") * dms.common.sel(regressor="HRF StimA")

f, ax = p.subplots(1, 1, figsize=(12, 3))
ax.plot(y.time, ts_ss.sel(window), "-", c="0.6", lw=1, label="data")
ax.plot(y.time, cleaned.sel(window), "r-", lw=1, label="data − nuisance")
ax.plot(y.time, true_hrf.sel(window), "k-", lw=2, label="true HRF")
vbx.plot_segments(ax, segments, fmt=SEGMENT_FMT)
ax.set_title(f"Ch. {ch}, HbO: nuisance signals removed")
ax.set_ylabel(r"$\Delta$ HbO / µM")
ax.set_xlabel("time / s")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left", ncol=3)
p.tight_layout()

## Things to keep in mind

- **Masking excludes samples, it does not repair them.** It is suited for transient
  artifacts like spikes and for unusable segments. `quality.id_motion` detects fast
  signal changes, so for a baseline shift only the jumps are marked, while the shifted
  segment stays in the fit. Baseline shifts should be corrected with a motion
  correction method, e.g. spline interpolation or TDDR, before fitting.
- **Avoid filtering data with artifacts before the fit.** A temporal filter spreads
  an artifact into the neighboring samples, which are not masked. Model slow drifts
  with drift regressors in the design matrix instead.
- **AR-IRLS and gaps.** The prewhitening filter of order $p$ combines each sample
  with the $p$ preceding samples. After each masked segment, the first $p$ filtered
  samples are therefore excluded as well. The AR coefficients are estimated from
  the residuals of the unmasked segments, joined end to end.
- **GLS** estimates the lag-1 autocorrelation from pairs of adjacent unmasked
  samples and needs at least two such pairs.
- **GLSAR** does not support masks with gaps between unmasked samples. Use AR-IRLS
  instead.
- **The mask must have the labels of the time series.** A mask computed for all
  channels must be reduced to the fitted channels first, e.g. with
  `mask.sel(channel=ts_long.channel)`.
- **The design matrix must be finite.** `glm.fit` raises an exception if a regressor
  contains NaNs or infs. `DesignMatrix.fill_nonfinite` replaces them and returns a
  mask that excludes the affected samples.
  The filled values also enter `glm.predict`. When subtracting a nuisance
  prediction from the data, set these samples to NaN with `.where(mask)`.

## References

In [ ]:
cedalion.bib.dump_to_notebook()